In [ ]:
import logging
import os
import sys
from pathlib import Path

import bioframe as bf
import cooler
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

sys.path.insert(1, "..")


from config.eda import DataConfig
from config.model_conf import FeaturesConfig, UnetConfig
from RNADNA_background.models.models import UnetNoiseModel
from RNADNA_background.utils.learn import (
    NoiseDatasetTracks,
    eval_epoch_unet,
    train_epoch_unet,
)

In [ ]:
# configuration of the dataset
data_conf_file = Path("../config/eda_conf.ini")
params = DataConfig(data_conf_file)

# configuration of the features
features_conf_file = Path("../config/data_conf.ini")
features_params = FeaturesConfig(features_conf_file, params.chromosomes)

model_conf_file = "../config/model_conf.ini"
model_params = UnetConfig(model_conf_file, "UnetNoiseModel")

# cool file to get bins coordinates
cool_file = params.hic_folder / Path(
    f"{params.cell_line}.mcool::resolutions/1000"
)
c = cooler.Cooler(str(cool_file))

logdir = Path("../logs")
logging.basicConfig(
    level=logging.INFO,
    filename=logdir / "unet_model.log",
    filemode="a",
    format="%(asctime)s %(levelname)s %(message)s",
)

In [ ]:
# loading data
windows = pd.read_csv(
    params.learn_data_folder
    / f"windows_{features_params.window_size}_{features_params.shift}.tsv",
    sep="\t",
)

features = pd.read_csv(
    params.learn_data_folder / f"features_{params.bin_size}.tsv",
    sep="\t",
    usecols=[
        "chrom",
        "start",
        "end",
        "bin",
        *features_params.num_features,
        *features_params.cat_features,
    ],
)

contacts = pd.read_csv(
    params.data_path / f"binned_contacts_{params.bin_size}.tsv",
    sep="\t",
)
print(contacts["count"].mean())
perc_99 = np.percentile(contacts["count"], features_params.perc_mask)
contacts.loc[contacts["count"] > perc_99, "count"] = perc_99

In [ ]:
# train val test split
train_windows = windows[
    windows["chr"].isin(features_params.train_chromosomes)
].reset_index(drop=True)
val_windows = windows[
    windows["chr"].isin(features_params.val_chromosomes)
].reset_index(drop=True)
test_windows = windows[
    windows["chr"].isin(features_params.test_chromosomes)
].reset_index(drop=True)

train_features = features[
    features["chrom"].isin(features_params.train_chromosomes)
]
val_features = features[
    features["chrom"].isin(features_params.val_chromosomes)
]
test_features = features[
    features["chrom"].isin(features_params.test_chromosomes)
]

train_contacts = contacts[
    contacts["dna_chr"].isin(features_params.train_chromosomes)
]
val_contacts = contacts[
    contacts["dna_chr"].isin(features_params.val_chromosomes)
]
test_contacts = contacts[
    contacts["dna_chr"].isin(features_params.test_chromosomes)
]

In [ ]:
# compute train means and stds
overlapped = bf.overlap(
    train_features,
    bf.merge(train_windows.rename({"chr": "chrom"}, axis=1), min_dist=0).drop(
        "n_intervals", axis=1
    ),
    how="left",
)

num_features_df = overlapped[~overlapped["chrom_"].isna()][
    features_params.num_features
]
means = num_features_df.mean()
stds = num_features_df.std()

In [ ]:
# creating dicts with tracks separated by chromosomes
train_features_dict = {
    chrom: features[features["chrom"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "chrom"], axis=1)
    for chrom in features_params.train_chromosomes
}
val_features_dict = {
    chrom: features[features["chrom"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "chrom"], axis=1)
    for chrom in features_params.val_chromosomes
}
test_features_dict = {
    chrom: features[features["chrom"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "chrom"], axis=1)
    for chrom in features_params.test_chromosomes
}

train_contacts_dict = {
    chrom: contacts[contacts["dna_chr"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "dna_chr"], axis=1)
    for chrom in features_params.train_chromosomes
}
val_contacts_dict = {
    chrom: contacts[contacts["dna_chr"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "dna_chr"], axis=1)
    for chrom in features_params.val_chromosomes
}
test_contacts_dict = {
    chrom: contacts[contacts["dna_chr"] == chrom]
    .set_index("bin")
    .drop(["start", "end", "dna_chr"], axis=1)
    for chrom in features_params.test_chromosomes
}

In [ ]:
train_ds = NoiseDatasetTracks(
    train_windows,
    train_features_dict,
    train_contacts_dict,
    features_params.num_features,
    means,
    stds,
    params.bin_size,
    mask_zeros=False,
)

val_ds = NoiseDatasetTracks(
    val_windows,
    val_features_dict,
    val_contacts_dict,
    features_params.num_features,
    means,
    stds,
    params.bin_size,
    mask_zeros=False,
    test=True,
)

test_ds = NoiseDatasetTracks(
    test_windows,
    test_features_dict,
    test_contacts_dict,
    features_params.num_features,
    means,
    stds,
    params.bin_size,
    mask_zeros=False,
    test=True,
)

In [ ]:
train_loader = DataLoader(
    train_ds, shuffle=True, batch_size=model_params.batch_size
)
val_loader = DataLoader(
    val_ds, shuffle=False, batch_size=model_params.batch_size
)

In [ ]:
device = "cuda"
model = UnetNoiseModel(
    kernel_size=model_params.kernel_size,
    features_n=len(features_params.num_features)
    + len(features_params.cat_features),
    channels=model_params.channels,
    activation=nn.SELU,
    dilation=model_params.dilations,
)

pytorch_total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
parameters_n = sum(p.numel() for p in model.parameters())
model = model.to(device)
criterion = nn.MSELoss(reduction="mean")
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=model_params.lr,
    weight_decay=model_params.weight_decay,
)
DIV_FACTOR = 50
max_lr = model_params.lr * DIV_FACTOR
sheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    div_factor=DIV_FACTOR,
    steps_per_epoch=len(train_loader),
    epochs=model_params.num_epochs,
)

In [ ]:
print(pytorch_total_params)

In [ ]:
val_losses = []
val_sccs = []
epochs_batch_loss = []
for i in range(model_params.num_epochs):
    train_loss = train_epoch_unet(
        model,
        train_loader,
        criterion,
        device,
        optimizer,
        sheduler,
        mask_zeros=False,
    )
    val_loss, val_scc, preds, targets, epoch_loss_list, val_epoch_scc = (
        eval_epoch_unet(model, val_loader, criterion, device, mask_zeros=False)
    )
    print(
        f"Epoch: {i+1}, Train loss: {train_loss:.3f}, Val loss: {val_loss:.3f}, Val SCC: {val_scc[0]:.3f}"
    )
    epochs_batch_loss.append(epoch_loss_list)
    val_losses.append(val_loss)
    val_sccs.append(val_scc[0])
path_to_model = Path(
    f"../models/{model._get_name()}_{params.experiment}_{params.cell_line}"
)
if not Path(path_to_model).is_dir():
    os.makedirs(path_to_model)
torch.save(model.state_dict(), path_to_model / "v1.pth")
logging.info(
    f"{params}, {model_params}, ONE CYCLE, DIV_FACTOR: {DIV_FACTOR}, SELU, train windows number: {len(train_ds)},  best val loss: {min(val_losses)}, best val scc: {max(val_sccs)}"
)